# Field Boundary Detection Pipeline - ResUNet
## End-to-End GeoAI Workflow

**Objective:** Detect agricultural field boundaries from RGB GeoTIFF using deep learning

**Input Files:**
- `BothavilleJMK22022.tif` (RGB raster, 380 MB, ~13,000 x 13,000 px)
- `Bothaville_fields.shp` (2,304 field polygons)

**Output Structure:**
```
Objective_1_Results/
├── predictions/          (boundary masks)
├── vector_outputs/       (shapefile polygons)
├── plots/                (visualizations)
├── model_weights/        (trained model)
├── metrics/              (evaluation results)
├── BothavilleJMK22022.tif     (input)
└── Bothaville_fields.shp      (input)
```

## STEP 1: IMPORTS & SETUP

In [ ]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from shapely.geometry import Polygon as ShapelyPolygon

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix as sklearn_cm

import cv2
from skimage.morphology import binary_closing, binary_opening
from skimage.measure import label, regionprops
from scipy.ndimage import binary_fill_holes

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ All imports successful")
print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

## STEP 2: SET PATHS & CREATE DIRECTORIES

In [ ]:
# ============================================
# IMPORTANT: Set your base folder path here
# ============================================
BASE_FOLDER = r"D:\MR_YAMBESA_RORO (D)\Objective_1_Results"

# Check if folder exists
if not os.path.exists(BASE_FOLDER):
    print(f"❌ ERROR: Folder does not exist: {BASE_FOLDER}")
    print("Please create the folder and place your TIF and shapefile there.")
    sys.exit(1)

# Define input file paths
INPUT_TIF = os.path.join(BASE_FOLDER, "BothavilleJMK22022.tif")
INPUT_SHP = os.path.join(BASE_FOLDER, "Bothaville_fields.shp")

# Check if input files exist
print("Checking input files...")
if not os.path.exists(INPUT_TIF):
    print(f"❌ TIF not found: {INPUT_TIF}")
    sys.exit(1)
else:
    tif_size = os.path.getsize(INPUT_TIF) / (1024**3)
    print(f"✓ TIF found: {INPUT_TIF} ({tif_size:.2f} GB)")

if not os.path.exists(INPUT_SHP):
    print(f"❌ Shapefile not found: {INPUT_SHP}")
    sys.exit(1)
else:
    print(f"✓ Shapefile found: {INPUT_SHP}")

# Create output directories
OUTPUT_DIRS = {
    'predictions': os.path.join(BASE_FOLDER, 'predictions'),
    'vectors': os.path.join(BASE_FOLDER, 'vector_outputs'),
    'plots': os.path.join(BASE_FOLDER, 'plots'),
    'models': os.path.join(BASE_FOLDER, 'model_weights'),
    'metrics': os.path.join(BASE_FOLDER, 'metrics')
}

for key, path in OUTPUT_DIRS.items():
    os.makedirs(path, exist_ok=True)
    print(f"✓ Output dir created: {path}")

print(f"\n✓ All paths configured successfully!")

## STEP 3: LOAD RASTER & SHAPEFILE

In [ ]:
print("Loading raster...")
with rasterio.open(INPUT_TIF) as src:
    img = src.read()  # (bands, height, width)
    transform = src.transform
    crs = src.crs
    height, width = src.height, src.width

print(f"✓ Raster loaded")
print(f"  Shape: {img.shape}")
print(f"  Resolution: {width} x {height} pixels")
print(f"  Bands: {img.shape[0]}")
print(f"  CRS: {crs}")

print("\nLoading shapefile...")
gdf = gpd.read_file(INPUT_SHP)
print(f"✓ Shapefile loaded")
print(f"  Number of fields: {len(gdf)}")
print(f"  CRS: {gdf.crs}")

# Reproject if needed
if gdf.crs != crs:
    print(f"  Reprojecting to {crs}...")
    gdf = gdf.to_crs(crs)
    print(f"  ✓ Reprojected")

## STEP 4: RASTERIZE POLYGONS TO GROUND TRUTH

In [ ]:
print("Rasterizing field polygons...")

# Convert polygons to raster format
geoms = [(geom, 1) for geom in gdf.geometry if geom is not None and geom.is_valid]

field_mask = rasterize(
    geoms,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype='uint8'
)

field_pixels = np.sum(field_mask > 0)
total_pixels = field_mask.shape[0] * field_mask.shape[1]
coverage = (field_pixels / total_pixels) * 100

print(f"✓ Polygons rasterized to {field_mask.shape}")
print(f"  Field pixels: {field_pixels:,} ({coverage:.2f}% coverage)")

## STEP 5: CREATE BOUNDARY GROUND TRUTH

In [ ]:
print("Generating boundary mask from field polygons...")

# Extract edges using Canny edge detector
field_uint = field_mask.astype(np.uint8)
edges = cv2.Canny((field_uint * 255), 50, 150)
boundary_mask = (edges > 0).astype(np.uint8)

# Dilate to make boundary thicker (better for training)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
boundary_mask = cv2.dilate(boundary_mask, kernel, iterations=1)

boundary_pixels = np.sum(boundary_mask > 0)

print(f"✓ Boundary mask created")
print(f"  Boundary pixels: {boundary_pixels:,} ({(boundary_pixels/total_pixels)*100:.2f}%)")

# Visualize reference
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(np.moveaxis(img, 0, -1)[:, :, :3] / 255.0)
axes[0].set_title('Original RGB Raster')
axes[0].axis('off')

axes[1].imshow(field_mask, cmap='gray')
axes[1].set_title('Rasterized Field Mask')
axes[1].axis('off')

axes[2].imshow(boundary_mask, cmap='gray')
axes[2].set_title('Ground Truth Boundary')
axes[2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIRS['plots'], '01_reference_data.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {os.path.join(OUTPUT_DIRS['plots'], '01_reference_data.png')}")

## STEP 6: NORMALIZE IMAGE

In [ ]:
print("Normalizing image...")

# Convert CHW to HWC
rgb = np.moveaxis(img, 0, -1).astype(np.float32)

# Normalize to [0, 1]
rgb_min = rgb.min()
rgb_max = rgb.max()
rgb = (rgb - rgb_min) / (rgb_max - rgb_min + 1e-8)

print(f"✓ Image normalized")
print(f"  Shape: {rgb.shape} (HWC format)")
print(f"  Value range: [{rgb.min():.4f}, {rgb.max():.4f}]")

## STEP 7: EXTRACT PATCHES

In [ ]:
print("Extracting patches from full raster...")

PATCH_SIZE = 256
STRIDE = 128  # 50% overlap

def extract_patches(image, label, patch_size=256, stride=128):
    """Extract overlapping patches from large images"""
    patches = []
    labels = []
    h, w = image.shape[0], image.shape[1]
    
    for y in range(0, h - patch_size + 1, stride):
        for x in range(0, w - patch_size + 1, stride):
            img_patch = image[y:y+patch_size, x:x+patch_size, :]
            lab_patch = label[y:y+patch_size, x:x+patch_size]
            
            if img_patch.shape[:2] == (patch_size, patch_size):
                patches.append(img_patch)
                labels.append(lab_patch)
    
    return patches, labels

img_patches, label_patches = extract_patches(rgb, boundary_mask, PATCH_SIZE, STRIDE)

print(f"✓ Patches extracted")
print(f"  Total patches: {len(img_patches)}")
print(f"  Patch size: {PATCH_SIZE} x {PATCH_SIZE}")
print(f"  Overlap: 50%")

## STEP 8: CREATE PYTORCH DATASET

In [ ]:
class FieldBoundaryDataset(Dataset):
    """PyTorch dataset for field boundary detection"""
    def __init__(self, images, masks):
        self.images = images
        self.masks = masks
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # HWC -> CHW
        img = self.images[idx].transpose(2, 0, 1).astype(np.float32)
        # Add channel dimension to mask
        msk = self.masks[idx].astype(np.float32)[None, :, :]
        
        return torch.from_numpy(img), torch.from_numpy(msk)

print("Creating train/val/test splits...")

# Split patches
indices = list(range(len(img_patches)))
train_idx, temp_idx = train_test_split(indices, train_size=0.7, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, train_size=0.5, random_state=42)

# Create datasets
train_dataset = FieldBoundaryDataset(
    [img_patches[i] for i in train_idx],
    [label_patches[i] for i in train_idx]
)
val_dataset = FieldBoundaryDataset(
    [img_patches[i] for i in val_idx],
    [label_patches[i] for i in val_idx]
)
test_dataset = FieldBoundaryDataset(
    [img_patches[i] for i in test_idx],
    [label_patches[i] for i in test_idx]
)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0)

print(f"✓ Datasets created")
print(f"  Train: {len(train_dataset)} patches")
print(f"  Val: {len(val_dataset)} patches")
print(f"  Test: {len(test_dataset)} patches")

## STEP 9: DEFINE RESUNET MODEL

In [ ]:
class ResidualBlock(nn.Module):
    """Residual block with skip connection"""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        
        self.skip = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )
    
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.skip(x)
        return self.relu(out)

class ResUNet(nn.Module):
    """ResUNet: Encoder-Decoder with residual blocks"""
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        
        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            ResidualBlock(32, 32)
        )
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.enc2 = nn.Sequential(
            ResidualBlock(32, 64),
            ResidualBlock(64, 64)
        )
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.enc3 = nn.Sequential(
            ResidualBlock(64, 128),
            ResidualBlock(128, 128)
        )
        self.pool3 = nn.MaxPool2d(2, 2)
        
        self.enc4 = nn.Sequential(
            ResidualBlock(128, 256),
            ResidualBlock(256, 256)
        )
        
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.MaxPool2d(2, 2),
            ResidualBlock(256, 512),
            ResidualBlock(512, 512),
            nn.ConvTranspose2d(512, 256, 2, stride=2)
        )
        
        # Decoder
        self.dec4 = nn.Sequential(
            ResidualBlock(512, 256),
            ResidualBlock(256, 256)
        )
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        
        self.dec3 = nn.Sequential(
            ResidualBlock(256, 128),
            ResidualBlock(128, 128)
        )
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        
        self.dec2 = nn.Sequential(
            ResidualBlock(128, 64),
            ResidualBlock(64, 64)
        )
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        
        self.dec1 = nn.Sequential(
            ResidualBlock(64, 32),
            ResidualBlock(32, 32)
        )
        
        self.final = nn.Sequential(
            nn.Conv2d(32, out_ch, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        
        # Bottleneck
        b = self.bottleneck(e4)
        
        # Decoder with skip
        d4 = self.dec4(torch.cat([b, e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        
        return self.final(d1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResUNet(in_ch=3, out_ch=1).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ ResUNet model created")
print(f"  Device: {device}")
print(f"  Total parameters: {total_params:,}")

## STEP 10: DEFINE LOSS & OPTIMIZER

In [ ]:
def dice_loss(pred, target, smooth=1.0):
    """Dice Loss"""
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    dice = (2.0 * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)
    return 1.0 - dice

def combined_loss(pred, target):
    """Dice + BCE"""
    bce = nn.BCELoss()(pred, target)
    dl = dice_loss(pred, target)
    return 0.5 * bce + 0.5 * dl

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

print("✓ Loss functions and optimizer configured")

## STEP 11: TRAIN MODEL

In [ ]:
print("Starting training...\n")

EPOCHS = 60
best_loss = float('inf')
patience = 15
patience_counter = 0

train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0.0
    for imgs, masks in train_loader:
        imgs = imgs.to(device).float()
        masks = masks.to(device).float()
        
        optimizer.zero_grad()
        preds = model(imgs)
        loss = combined_loss(preds, masks)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(device).float()
            masks = masks.to(device).float()
            preds = model(imgs)
            loss = combined_loss(preds, masks)
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    # Early stopping
    if val_loss < best_loss:
        best_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIRS['models'], 'best_model.pth'))
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

print(f"\n✓ Training complete")
print(f"  Best val loss: {best_loss:.4f}")
print(f"  Model saved: {os.path.join(OUTPUT_DIRS['models'], 'best_model.pth')}")

## STEP 12: PLOT TRAINING CURVES

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Val Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIRS['plots'], '02_training_curves.png'), dpi=150)
plt.show()

print(f"✓ Saved: {os.path.join(OUTPUT_DIRS['plots'], '02_training_curves.png')}")

## STEP 13: INFERENCE ON FULL RASTER

In [ ]:
print("Loading best model...")
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIRS['models'], 'best_model.pth'), map_location=device))
model.eval()

print("Running inference on full raster...")
full_pred = np.zeros((rgb.shape[0], rgb.shape[1]), dtype=np.float32)

with torch.no_grad():
    count = 0
    total = len(range(0, rgb.shape[0] - PATCH_SIZE + 1, STRIDE)) * len(range(0, rgb.shape[1] - PATCH_SIZE + 1, STRIDE))
    
    for y in range(0, rgb.shape[0] - PATCH_SIZE + 1, STRIDE):
        for x in range(0, rgb.shape[1] - PATCH_SIZE + 1, STRIDE):
            patch = rgb[y:y+PATCH_SIZE, x:x+PATCH_SIZE, :]
            patch_t = torch.from_numpy(patch.transpose(2, 0, 1)[None, :, :, :]).float().to(device)
            pred = model(patch_t)
            pred_map = pred[0, 0].cpu().numpy()
            full_pred[y:y+PATCH_SIZE, x:x+PATCH_SIZE] = np.maximum(
                full_pred[y:y+PATCH_SIZE, x:x+PATCH_SIZE],
                pred_map
            )
            count += 1
            if count % 20 == 0:
                print(f"  Progress: {count}/{total}")

print(f"\n✓ Inference complete")
print(f"  Prediction map shape: {full_pred.shape}")
print(f"  Value range: [{full_pred.min():.4f}, {full_pred.max():.4f}]")

## STEP 14: POST-PROCESSING

In [ ]:
print("Post-processing...")

# Threshold
pred_binary = full_pred >= 0.5

# Morphological operations
pred_binary = binary_opening(pred_binary)
pred_binary = binary_closing(pred_binary)
pred_binary = binary_fill_holes(pred_binary)

# Remove small objects
label_img = label(pred_binary)
props = regionprops(label_img)
filtered = np.zeros_like(pred_binary, dtype=np.uint8)
for region in props:
    if region.area >= 50:
        filtered[label_img == region.label] = 1

pred_binary = filtered.astype(np.uint8)

print(f"✓ Post-processing complete")
print(f"  Final objects: {len(props)}")

## STEP 15: VISUALIZE RESULTS

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Full images
axes[0, 0].imshow(np.moveaxis(img, 0, -1)[:, :, :3] / 255.0)
axes[0, 0].set_title('RGB Raster')
axes[0, 0].axis('off')

axes[0, 1].imshow(boundary_mask, cmap='gray')
axes[0, 1].set_title('Ground Truth Boundary')
axes[0, 1].axis('off')

axes[0, 2].imshow(pred_binary, cmap='gray')
axes[0, 2].set_title('Predicted Boundary')
axes[0, 2].axis('off')

# Zoomed details
sl = slice(500, 1000)
axes[1, 0].imshow(np.moveaxis(img, 0, -1)[sl, sl, :3] / 255.0)
axes[1, 0].set_title('RGB Detail')
axes[1, 0].axis('off')

axes[1, 1].imshow(boundary_mask[sl, sl], cmap='gray')
axes[1, 1].set_title('GT Detail')
axes[1, 1].axis('off')

axes[1, 2].imshow(pred_binary[sl, sl], cmap='gray')
axes[1, 2].set_title('Pred Detail')
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIRS['plots'], '03_prediction_results.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {os.path.join(OUTPUT_DIRS['plots'], '03_prediction_results.png')}")

## STEP 16: SAVE PREDICTION MASKS

In [ ]:
print("Saving prediction masks...")

# Save as numpy
np.save(os.path.join(OUTPUT_DIRS['predictions'], 'boundary_mask.npy'), pred_binary)

# Save as GeoTIFF
with rasterio.open(
    os.path.join(OUTPUT_DIRS['predictions'], 'boundary_mask.tif'),
    'w',
    driver='GTiff',
    height=pred_binary.shape[0],
    width=pred_binary.shape[1],
    count=1,
    dtype=pred_binary.dtype,
    crs=crs,
    transform=transform
) as dst:
    dst.write(pred_binary, 1)

# Save as PNG
plt.imsave(os.path.join(OUTPUT_DIRS['predictions'], 'boundary_mask.png'), pred_binary, cmap='gray')

print(f"✓ Prediction masks saved to {OUTPUT_DIRS['predictions']}/")

## STEP 17: VECTORIZE TO POLYGONS

In [ ]:
print("Vectorizing boundary mask...")

contours, _ = cv2.findContours(pred_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

polygons = []
for cnt in contours:
    if len(cnt) >= 3:
        pts = [(int(p[0][0]), int(p[0][1])) for p in cnt]
        try:
            poly = ShapelyPolygon(pts)
            if poly.is_valid and poly.area > 0:
                polygons.append(poly)
        except:
            pass

gdf_pred = gpd.GeoDataFrame({'geometry': polygons}, crs=crs)

# Save as shapefile
gdf_pred.to_file(os.path.join(OUTPUT_DIRS['vectors'], 'predicted_boundaries.shp'))

# Save as GeoJSON
gdf_pred.to_file(os.path.join(OUTPUT_DIRS['vectors'], 'predicted_boundaries.geojson'), driver='GeoJSON')

print(f"✓ Vectorized {len(gdf_pred)} polygons")
print(f"  Saved to {OUTPUT_DIRS['vectors']}/")

## STEP 18: COMPUTE METRICS

In [ ]:
print("Computing evaluation metrics...\n")

ref = boundary_mask.astype(np.uint8)
pred = pred_binary.astype(np.uint8)

# Pixel-level
tp = np.sum((pred == 1) & (ref == 1))
fp = np.sum((pred == 1) & (ref == 0))
fn = np.sum((pred == 0) & (ref == 1))
tn = np.sum((pred == 0) & (ref == 0))

precision = tp / (tp + fp + 1e-8)
recall = tp / (tp + fn + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)
iou = tp / (tp + fp + fn + 1e-8)
dice = 2 * tp / (2 * tp + fp + fn + 1e-8)

metrics = {
    'precision': float(precision),
    'recall': float(recall),
    'f1': float(f1),
    'iou': float(iou),
    'dice': float(dice),
    'tp': int(tp),
    'fp': int(fp),
    'fn': int(fn),
    'tn': int(tn)
}

print("PIXEL-LEVEL METRICS:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")
print(f"  IoU:       {iou:.4f}")
print(f"  Dice:      {dice:.4f}")

# Save metrics
with open(os.path.join(OUTPUT_DIRS['metrics'], 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"\n✓ Metrics saved to {OUTPUT_DIRS['metrics']}/")

## STEP 19: CONFUSION MATRIX

In [ ]:
cm = sklearn_cm(ref.flatten(), pred.flatten())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Background', 'Boundary'],
            yticklabels=['Background', 'Boundary'])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('True')
axes[0].set_xlabel('Predicted')

# Metrics bar chart
metric_names = ['Precision', 'Recall', 'F1', 'IoU', 'Dice']
metric_vals = [precision, recall, f1, iou, dice]
bars = axes[1].bar(metric_names, metric_vals, color='steelblue')
axes[1].set_ylabel('Score')
axes[1].set_ylim([0, 1])
axes[1].grid(alpha=0.3, axis='y')

for bar, val in zip(bars, metric_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.3f}', ha='center', va='bottom')

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIRS['plots'], '04_metrics.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved: {os.path.join(OUTPUT_DIRS['plots'], '04_metrics.png')}")

## STEP 20: SUMMARY

In [ ]:
print("\n" + "="*70)
print(" PIPELINE COMPLETE")
print("="*70)

print("\n📊 INPUT:")
print(f"  Raster: {width} x {height} px ({img.shape[0]} bands)")
print(f"  Fields: {len(gdf)} polygons")

print("\n🎯 TRAINING:")
print(f"  Patches: {len(img_patches)}")
print(f"  Train/Val/Test: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}")
print(f"  Epochs trained: {len(train_losses)}")
print(f"  Best val loss: {best_loss:.4f}")

print("\n📈 RESULTS:")
print(f"  Precision:  {precision:.4f}")
print(f"  Recall:     {recall:.4f}")
print(f"  F1-Score:   {f1:.4f}")
print(f"  IoU:        {iou:.4f}")
print(f"  Dice:       {dice:.4f}")

print("\n📁 OUTPUT FILES:")
for key, path in OUTPUT_DIRS.items():
    files = os.listdir(path)
    print(f"\n  {key.upper()}:")
    for f in files:
        print(f"    ✓ {f}")

print("\n" + "="*70)
print(f" All outputs saved to: {BASE_FOLDER}")
print("="*70)